In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# 1. Imports et chargement des données
# ============================================================
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error

# Chargement des fichiers
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
train['date'] = pd.to_datetime(train['date'])

# ============================================================
# 2. Feature engineering
# ============================================================
def create_features(df):
    df = df.copy()
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    return df

train = create_features(train)

# ============================================================
# 3. Création des lags
# ============================================================
def create_lags(df):
    df = df.sort_values(['store_nbr', 'family', 'date'])
    df['lag_1'] = df.groupby(['store_nbr', 'family'])['sales'].shift(1)
    df['lag_7'] = df.groupby(['store_nbr', 'family'])['sales'].shift(7)
    df['rolling_mean_7'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean()
    )
    return df

train = create_lags(train)
train = train.fillna(0)

# ============================================================
# 4. Séparation train / validation
# ============================================================
train_features = train[train['date'] <= '2017-07-31'].copy()
valid_features = train[(train['date'] >= '2017-08-01') & (train['date'] <= '2017-08-15')].copy()

# ============================================================
# 5. Features pour le modèle global
# ============================================================
features = ['dayofweek', 'month', 'year', 'lag_1', 'lag_7', 'rolling_mean_7', 'store_nbr', 'family']

X_train = pd.get_dummies(train_features[features], drop_first=True)
X_valid = pd.get_dummies(valid_features[features], drop_first=True)

# Aligner les colonnes
X_train, X_valid = X_train.align(X_valid, join='left', axis=1, fill_value=0)

y_train = train_features['sales']
y_valid = valid_features['sales']

# ============================================================
# 6. Modèle linéaire global
# ============================================================
model = LinearRegression()
print("entrainement du model ... ")
model.fit(X_train, y_train)

y_pred = model.predict(X_valid)
y_pred = np.maximum(y_pred, 0)  # pas de ventes négatives

# RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_valid, y_pred))
print("✅ RMSLE linéaire global avec One-Hot :", rmsle)

# Moyenne pour check
print("Moyenne ventes réelles :", y_valid.mean())
print("Moyenne ventes prédites :", y_pred.mean())
#----------------------------------------------------------------------------------------

from xgboost import XGBRegressor

# --- 1. Définir la période pour entraîner le XGBoost sur les résidus ---
xgb_train_period = train[(train['date'] >= '2017-04-01') & (train['date'] <= '2017-07-31')].copy()

# --- 2. Features pour XGBoost ---
X_XGB = pd.get_dummies(xgb_train_period[features], drop_first=True)

# Aligner les colonnes avec le X_train du modèle linéaire
X_XGB, _ = X_XGB.align(X_train, join='right', axis=1, fill_value=0)

# --- 3. Prédictions linéaires sur cette période ---
y_pred_lin_XGB = model.predict(X_XGB)
y_pred_lin_XGB = np.maximum(y_pred_lin_XGB, 0)

# --- 4. Calcul des résidus ---
y_resid = xgb_train_period['sales'].values - y_pred_lin_XGB

# --- 5. Entraînement XGBoost sur les résidus ---
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("✅ Entraînement XGBoost sur résidus...")
xgb_model.fit(X_XGB, y_resid)

# --- 6. Prédiction des résidus sur la validation ---
X_valid_aligned = X_valid.copy()
X_valid_aligned, _ = X_valid_aligned.align(X_train, join='right', axis=1, fill_value=0)

y_pred_resid_valid = xgb_model.predict(X_valid_aligned)

# --- 7. Prédiction finale hybride ---
y_pred_final = np.maximum(y_pred + y_pred_resid_valid, 0)

# --- 8. RMSLE final ---
rmsle_hybrid = np.sqrt(mean_squared_log_error(y_valid, y_pred_final))
print("✅ RMSLE modèle hybride (Linéaire + XGBoost résidus) :", rmsle_hybrid)

# Moyennes pour vérification
print("Moyenne ventes réelles :", y_valid.mean())
print("Moyenne ventes prédites :", y_pred_final.mean())


In [ ]:
import pandas as pd
import numpy as np

# --- Chargement du test set ---
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")
test['date'] = pd.to_datetime(test['date'])

# --- Préparer les lag features à partir de la période 26 juillet - 10 août ---
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
train['date'] = pd.to_datetime(train['date'])

source_start = '2017-07-26'
source_end = '2017-08-10'
source = train[(train['date'] >= source_start) & (train['date'] <= source_end)].copy()
source = source.sort_values(['store_nbr', 'family', 'date'])

# --- On crée les lag features pour le test set ---
lag_1_dict = {}
lag_7_dict = {}
rolling_mean_7_dict = {}

for store in source['store_nbr'].unique():
    for fam in source['family'].unique():
        temp = source[(source['store_nbr'] == store) & (source['family'] == fam)].sort_values('date')
        lag_1_dict[(store, fam)] = temp['sales'].values[-1]          # dernier jour = lag_1
        lag_7_dict[(store, fam)] = temp['sales'].values[0]           # premier jour de la semaine = lag_7
        rolling_mean_7_dict[(store, fam)] = temp['sales'].mean()     # moyenne sur la semaine

# --- Ajouter les features au test set ---
test['lag_1'] = test.apply(lambda x: lag_1_dict.get((x['store_nbr'], x['family']), 0), axis=1)
test['lag_7'] = test.apply(lambda x: lag_7_dict.get((x['store_nbr'], x['family']), 0), axis=1)
test['rolling_mean_7'] = test.apply(lambda x: rolling_mean_7_dict.get((x['store_nbr'], x['family']), 0), axis=1)

# --- Feature engineering ---
test['dayofweek'] = test['date'].dt.dayofweek
test['month'] = test['date'].dt.month
test['year'] = test['date'].dt.year

# --- Colonnes pour le modèle ---
features = ['dayofweek', 'month', 'year', 'lag_1', 'lag_7', 'rolling_mean_7', 'store_nbr', 'family']

X_test = pd.get_dummies(test[features], drop_first=True)

# Aligner avec les colonnes du train utilisé pour le linear et xgb
X_test, _ = X_test.align(X_train, join='right', axis=1, fill_value=0)

# --- Prédiction ---
y_pred_lin_test = np.maximum(model.predict(X_test), 0)
y_pred_resid_test = xgb_model.predict(X_test)

y_pred_final_test = np.maximum(y_pred_lin_test + y_pred_resid_test, 0)

# --- Création de la submission ---
submission = test[['id']].copy()
submission['sales'] = y_pred_final_test
submission.to_csv("submission_hybrid_final.csv", index=False)
print("✅ submission_hybrid_final.csv créée !")


In [ ]:
print(submission)